# Advanced Evaluation for AI-SOC Models

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/processed/unsw_nb15_processed.csv")

X = df.drop(columns=["label"])
y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
rf = joblib.load("../models/random_forest.pkl")
iso = joblib.load("../models/isolation_forest.pkl")

In [ ]:
rf_probs = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_probs)
rf_auc

In [ ]:
iso_score = iso.decision_funtion(X_test)
iso_auc = roc_auc_score(y_test, -iso_scores)
iso_auc

In [ ]:
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
iso_fpr, iso_tpr, _ = roc_curve(y_test, -iso_scores)

plt.figure(figsize=(6,5))
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC={rf_auc:.2f})")
plt.plot(iso_fpr, iso_tpr, label=f"Isolation Forest (AUC={iso_auc:.2f})")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison for AI-SOC")
plt.legend()
plt.show()

In [ ]:
def false_positive_rate(y_true, y_pred) :
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn)

In [ ]:
rf_preds = rf.predict(X_test)
iso_preds = np.where(iso.predict(X_test) == -1, 1, 0)

rf_fpr, iso_fpr

In [ ]:
test_results = X_test.copy()
test_results["actual"] = y_test.values
test_results["rf_detected"] = rf_preds

In [ ]:
attack_events = test_results[test_results["actual"] == 1]

first_detected_index = attack_events[
    attack_events["rf_detected"] == 1
].index.min()

mttd_simulated = first_detected_index - attack_events.index.min()
mttd_simulated

In [ ]:
summary = pd.DataFrame({
    "Metric": ["ROC-AUC", "False Positive Rate", "Detection capability"],
    "Random Forest": [rf_auc, rf_fpr, "Known Attacks"],
    "Isolation Forest": [iso_auc, iso_fpr, "Unknown / Zero-day"]
})

summary